# Demo 3 — MLflow Technique Comparison via `mlflow.genai.evaluate`
## Session 6: Advanced Prompt Engineering

> **MLflow as the tape recorder.** Every prompt sent, every response returned, every tool call, every latency and token count — captured automatically by `mlflow.openai.autolog()` and the `@mlflow.trace` decorators below. Run the cells, then open the MLflow UI to replay every input and output the model saw.

**Goal:** Stop debating techniques. Let MLflow rank them on a frozen eval set.

**Task:** Root-cause analysis (RCA) on production log snippets.

**What we'll compare (same task, same model, same eval set):**
1. **Zero-shot CoT** — single call, "Let's think step by step"
2. **Plan-and-Solve** — single call, plan-then-execute (free upgrade over CoT)
3. **Self-Refine (2 rounds)** — generator → critic → refiner loop
4. **Tree-of-Thoughts (beam=2, depth=2)** — propose-and-evaluate BFS

**Scorers:** Built-in `Correctness`, `Guidelines`, `Safety` + custom `cites_evidence`, `cost_under_5_cents`.

**What to watch:** The MLflow Compare view ranking runs by `Correctness/mean` — numbers decide architecture.

---
**Pre-session setup:** configure via env vars (see Setup cell). For local dev: `mlflow ui --port 5000`.

## Setup — env-var driven

```bash
export MLFLOW_TRACKING_URI=http://127.0.0.1:5000             # or Databricks workspace URL
export MLFLOW_EXPERIMENT=session6/demo_03_technique_comparison
export OPENAI_BASE_URL=https://api.openai.com/v1                 # OpenAI-compatible endpoint
export OPENAI_API_KEY=sk-...                                      # leave unset for MOCK mode
export MODEL=gpt-4o-mini
```

`mlflow.openai.autolog()` makes every LLM call self-documenting in the trace viewer.

**Local-by-default.** Notebook ships with `OPENAI_BASE_URL=http://localhost:11434/v1` and `MODEL=qwen2.5-coder:7b` (Ollama). Set the env vars to point at OpenAI, Anthropic, vLLM, Together, etc. — any OpenAI-compatible endpoint.

**Cost tracking uses hypothetical local-LLM rates** (~$0.00005/1K tokens) so the cost-aware traces and scorers still produce meaningful numbers in the demos. Override the `PRICING` dict with your real per-1K rates for production.

In [ ]:
# -- Setup -----------------------------------------------------------------
import os, json, re, time, hashlib
import mlflow
from mlflow.entities import SpanType
from mlflow.genai.scorers import scorer, Correctness, Guidelines, Safety, RelevanceToQuery
from openai import OpenAI

# ─── MLflow tracking ─────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI      = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
MLFLOW_TRACKING_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME")  # optional (Databricks/remote)
MLFLOW_TRACKING_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD")  # optional
MLFLOW_REGISTRY_URI      = os.getenv("MLFLOW_REGISTRY_URI", MLFLOW_TRACKING_URI)
MLFLOW_EXPERIMENT        = os.getenv("MLFLOW_EXPERIMENT", "session6/demo_03_technique_comparison")

if MLFLOW_TRACKING_USERNAME:
    os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
if MLFLOW_TRACKING_PASSWORD:
    os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_registry_uri(MLFLOW_REGISTRY_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

# ─── LLM (OpenAI-compatible: OpenAI, Azure, vLLM, Ollama, Together, etc.) ────
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "http://localhost:11434/v1")  # Ollama default
OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY", "ollama")  # Ollama ignores; any non-empty string works
MODEL    = os.getenv("MODEL", "qwen2.5-coder:7b")  # local-default for technique comparison

# Reproducibility default for the eval comparison: same temperature for every technique.
# (ToT propose step intentionally lifts to 0.7 for diversity — search needs spread.)
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.0"))

client = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY or "no-key")

# ─── Turn MLflow into the tape recorder ──────────────────────────────────────
# Captures every chat.completions.create call — request, response, tokens, latency, cost.
mlflow.openai.autolog()

USE_MOCK = not OPENAI_API_KEY

# Per-1K-token pricing (USD) — update at each model release.
PRICING = {
    "gpt-4o-mini":        {"in": 0.00015, "out": 0.0006},
    "gpt-4o":             {"in": 0.0025,  "out": 0.010},
    "o1-mini":            {"in": 0.003,   "out": 0.012},
    "claude-sonnet-4-6":  {"in": 0.003,   "out": 0.015},
    # Hypothetical local-LLM pricing (electricity + amortised GPU per 1K tokens — adjust to your infra)
    "qwen2.5-coder:14b":  {"in": 0.00005, "out": 0.00005},
    "qwen2.5-coder:7b":   {"in": 0.00002, "out": 0.00002},
    "llama3.1:8b":        {"in": 0.00002, "out": 0.00002},
}

def price_of(model: str, usage) -> float:
    """USD cost for one chat call. 0.0 in mock mode.
    Unknown models fall back to hypothetical local-LLM rates (~$0.00005/1K)."""
    if usage is None:
        return 0.0
    p = PRICING.get(model, {"in": 0.00005, "out": 0.00005})  # hypothetical fallback
    return usage.prompt_tokens / 1000 * p["in"] + usage.completion_tokens / 1000 * p["out"]

print(f"MLflow:  {MLFLOW_TRACKING_URI}  (exp: {MLFLOW_EXPERIMENT})")
print(f"LLM:     {OPENAI_BASE_URL}  model={MODEL}  mock={USE_MOCK}  temperature={TEMPERATURE}")
print(f"UI:      open {MLFLOW_TRACKING_URI}  → Experiments → {MLFLOW_EXPERIMENT}")
print(f"NOTE: defaults to local Ollama (http://localhost:11434/v1). Set OPENAI_BASE_URL/OPENAI_API_KEY/MODEL to point at OpenAI/Anthropic/vLLM.")

## The golden eval set

Five realistic RCA scenarios. Each row carries the log snippet + symptoms (`inputs.report`) and the gold root cause (`expected_response`). The built-in `Correctness` judge compares the technique's output against the gold answer.

In [ ]:
eval_data = [
    {
        "inputs": {"report": (
            "[09:14:22] api ERROR: HikariCP - Connection is not available, request timed out after 30000ms.\n"
            "[09:14:23] api WARN: pool size=20, active=20, idle=0, waiting=147.\n"
            "Symptoms: 5xx spike on /checkout starting 09:14, p99 latency 30s, traffic flat vs baseline."
        )},
        "expected_response": (
            "Root cause: db connection pool exhaustion on api service at 09:14:22. "
            "HikariCP pool of 20 fully saturated with 147 waiters. Mitigation: raise pool size and "
            "investigate long-running queries holding connections."
        ),
    },
    {
        "inputs": {"report": (
            "[03:02:11] worker INFO: GC pause Full (Allocation Failure) 8421ms.\n"
            "[03:02:20] worker WARN: heap after GC = 7.8G / 8G.\n"
            "Symptoms: kafka consumer lag climbing on payments topic, no deploys in 48h."
        )},
        "expected_response": (
            "Root cause: long GC pause on worker (8.4s at 03:02:11) due to heap near capacity (7.8G/8G). "
            "Stop-the-world pause blocks kafka consumer, causing lag. Mitigation: raise heap or fix memory leak."
        ),
    },
    {
        "inputs": {"report": (
            "[14:55:01] ingress ERROR: upstream timed out (110: Connection timed out) while connecting to upstream, service=api.\n"
            "[14:55:02] api INFO: outbound call to redis took 4900ms (timeout 5000ms).\n"
            "Symptoms: ingress 504s on /feed, api healthy in isolation, redis CPU=98%."
        )},
        "expected_response": (
            "Root cause: redis saturation at 14:55:01 (CPU 98%) causes api outbound calls to approach timeout, "
            "cascading to ingress 504s on /feed. Mitigation: scale redis or cache hot keys at api layer."
        ),
    },
    {
        "inputs": {"report": (
            "[11:30:00] deploy INFO: api rolled out version v4.7.\n"
            "[11:33:42] api ERROR: java.lang.OutOfMemoryError: Java heap space.\n"
            "[11:33:43] kubelet INFO: pod api-7f9 OOMKilled.\n"
            "Symptoms: api crashloop starting 11:33, last clean deploy v4.6 at 09:00."
        )},
        "expected_response": (
            "Root cause: api v4.7 deployed at 11:30 introduced an OOM — heap exhausted at 11:33:42, pod "
            "OOMKilled. Mitigation: roll back to v4.6 immediately; profile v4.7 for memory regression."
        ),
    },
    {
        "inputs": {"report": (
            "[00:00:03] api ERROR: javax.net.ssl.SSLHandshakeException: PKIX path validation failed: "
            "java.security.cert.CertPathValidatorException: validity check failed.\n"
            "Symptoms: api fails all outbound calls to payments-gateway since midnight, no deploys in 7 days."
        )},
        "expected_response": (
            "Root cause: TLS certificate for payments-gateway expired at midnight (00:00:03), causing "
            "PKIX validation failure on api outbound calls. Mitigation: renew certificate and add expiry "
            "monitoring 30 days before due date."
        ),
    },
]
print(f"Loaded {len(eval_data)} RCA cases.")

## The four techniques

Each technique is a function `report -> answer`. All four are decorated with `@mlflow.trace` so MLflow captures the full span tree per call. Inside each technique we attach `cost_usd` as a trace tag so the `cost_under_5_cents` scorer can read it back later.

In [ ]:
def _mock_answer(prefix: str, report: str) -> tuple[str, float]:
    """Deterministic mock: pulls timestamp + service from the log so cites_evidence passes."""
    ts = re.search(r"\d{2}:\d{2}:\d{2}", report)
    svc_m = re.search(r"\b(api|db|kafka|redis|worker|ingress|deploy|kubelet)\b", report, re.I)
    ts = ts.group(0) if ts else "00:00:00"
    svc = svc_m.group(0) if svc_m else "api"
    # Simulated cost varies by technique to make the demo numbers interesting
    cost_lookup = {"zero_shot_cot": 0.0009, "plan_and_solve": 0.0011,
                   "self_refine": 0.0040, "tot": 0.0150}
    cost = cost_lookup.get(prefix, 0.001)
    answer = (
        f"[{prefix}] At {ts} the {svc} service showed the failure signal. "
        f"Likely root cause involves {svc} saturation or regression. "
        f"Mitigation: investigate {svc} and roll back or scale as appropriate."
    )
    return answer, cost

def _chat(messages, temperature: float | None = None, technique_tag: str = ""):
    """Single LLM call — returns (text, cost_usd, latency_ms). Threads TEMPERATURE by default."""
    if temperature is None:
        temperature = TEMPERATURE
    if USE_MOCK:
        return None, 0.0, 0.0
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=temperature, max_tokens=400,
    )
    latency_ms = (time.time() - t0) * 1000
    cost = price_of(MODEL, resp.usage)
    return resp.choices[0].message.content, cost, latency_ms

# Track running totals so we can tag the parent trace with both cost AND latency.
_running_latency_ms = {"v": 0.0}

def _accumulate_cost(running: float, delta: float, latency_ms: float = 0.0) -> float:
    """Add `delta` to running cost and bump trace tag. Latency arg also bumps a latency tag."""
    new_total = running + delta
    _running_latency_ms["v"] += latency_ms
    span = mlflow.get_current_active_span()
    if span:
        span.set_attribute("cost_usd", new_total)
        span.set_attribute("latency_ms", _running_latency_ms["v"])
    try:
        mlflow.update_current_trace(tags={
            "cost_usd":   f"{new_total:.6f}",
            "latency_ms": f"{_running_latency_ms['v']:.0f}",
        })
    except Exception:
        pass
    return new_total

def _reset_running_latency() -> None:
    _running_latency_ms["v"] = 0.0

In [ ]:
@mlflow.trace(name="zero_shot_cot", span_type=SpanType.CHAIN,
              attributes={"technique": "zero_shot_cot"})
def zero_shot_cot(report: str) -> str:
    _reset_running_latency()
    # Tape-recorder: capture the eval-row input on the parent CHAIN span.
    span = mlflow.get_current_active_span()
    if span:
        span.set_inputs({"prompt": report, "params": {"model": MODEL, "technique": "zero_shot_cot",
                                                       "temperature": TEMPERATURE}})
    if USE_MOCK:
        ans, cost = _mock_answer("zero_shot_cot", report)
        _accumulate_cost(0.0, cost, latency_ms=0.0)
        if span: span.set_outputs({"text": ans, "cost_usd": cost})
        return ans
    sys = "You are a senior SRE doing RCA from a log snippet. Let's think step by step."
    user = f"Log snippet and symptoms:\n{report}\n\nStep-by-step reasoning, then a 3-sentence root cause + mitigation."
    text, cost, latency_ms = _chat([{"role": "system", "content": sys},
                                     {"role": "user",   "content": user}], technique_tag="zero_shot_cot")
    _accumulate_cost(0.0, cost, latency_ms=latency_ms)
    if span: span.set_outputs({"text": text, "cost_usd": cost})
    return text

In [ ]:
@mlflow.trace(name="plan_and_solve", span_type=SpanType.CHAIN,
              attributes={"technique": "plan_and_solve"})
def plan_and_solve(report: str) -> str:
    _reset_running_latency()
    span = mlflow.get_current_active_span()
    if span:
        span.set_inputs({"prompt": report, "params": {"model": MODEL, "technique": "plan_and_solve",
                                                       "temperature": TEMPERATURE}})
    if USE_MOCK:
        ans, cost = _mock_answer("plan_and_solve", report)
        _accumulate_cost(0.0, cost, latency_ms=0.0)
        if span: span.set_outputs({"text": ans, "cost_usd": cost})
        return ans
    sys = "You are a senior SRE doing RCA from a log snippet."
    user = (
        "First, understand the problem and devise a plan. "
        "Then execute the plan, paying attention to variables (timestamps, services, error codes).\n\n"
        f"Log snippet and symptoms:\n{report}\n\n"
        "End with a 3-sentence answer: root-cause service, key timestamp, concrete mitigation."
    )
    text, cost, latency_ms = _chat([{"role": "system", "content": sys},
                                     {"role": "user",   "content": user}], technique_tag="plan_and_solve")
    _accumulate_cost(0.0, cost, latency_ms=latency_ms)
    if span: span.set_outputs({"text": text, "cost_usd": cost})
    return text

In [ ]:
@mlflow.trace(name="self_refine", span_type=SpanType.CHAIN,
              attributes={"technique": "self_refine", "max_rounds": 2})
def self_refine(report: str, max_rounds: int = 2) -> str:
    """Generator → critic → refiner, capped at 2 rounds. Exit early if critic says 'no changes'."""
    _reset_running_latency()
    root_span = mlflow.get_current_active_span()
    if root_span:
        root_span.set_inputs({"prompt": report,
                              "params": {"model": MODEL, "technique": "self_refine",
                                          "max_rounds": max_rounds, "temperature": TEMPERATURE}})
    if USE_MOCK:
        ans, cost = _mock_answer("self_refine", report)
        _accumulate_cost(0.0, cost, latency_ms=0.0)
        if root_span: root_span.set_outputs({"text": ans, "cost_usd": cost})
        return ans
    sys = "You are a senior SRE doing RCA from a log snippet."
    running_cost = 0.0

    # Round 0 — initial generation
    with mlflow.start_span(name="generate", span_type=SpanType.LLM) as gen_sp:
        gen_prompt = f"Log:\n{report}\n\nWrite a 3-sentence RCA: service, timestamp, mitigation."
        gen_sp.set_inputs({"prompt": gen_prompt, "params": {"model": MODEL, "round": 0,
                                                              "temperature": TEMPERATURE}})
        draft, c, lat = _chat([
            {"role": "system", "content": sys},
            {"role": "user",   "content": gen_prompt},
        ])
        running_cost = _accumulate_cost(running_cost, c, latency_ms=lat)
        gen_sp.set_attribute("latency_ms", lat)
        gen_sp.set_attribute("cost_usd", c)
        gen_sp.set_outputs({"text": draft, "cost_usd": c, "latency_ms": lat})

    for r in range(1, max_rounds + 1):
        with mlflow.start_span(name=f"critique_{r}", span_type=SpanType.LLM) as crit_sp:
            crit_prompt = (
                f"Original log:\n{report}\n\nDraft RCA:\n{draft}\n\n"
                "Critique: cite any missing timestamp, missing service, or weak mitigation. "
                "If the draft is correct and complete, reply exactly 'no changes'."
            )
            crit_sp.set_inputs({"prompt": crit_prompt, "params": {"model": MODEL, "round": r,
                                                                    "draft": draft, "temperature": TEMPERATURE}})
            crit, c, lat = _chat([
                {"role": "system", "content": "You are a strict reviewer of RCA reports."},
                {"role": "user",   "content": crit_prompt},
            ])
            running_cost = _accumulate_cost(running_cost, c, latency_ms=lat)
            crit_sp.set_attribute("latency_ms", lat)
            crit_sp.set_attribute("cost_usd", c)
            crit_sp.set_outputs({"text": crit, "cost_usd": c, "latency_ms": lat})
        if crit and crit.lower().strip().startswith("no changes"):
            mlflow.update_current_trace(tags={"stop_reason": "critic_satisfied", "rounds_used": str(r - 1)})
            if root_span: root_span.set_outputs({"text": draft, "cost_usd": running_cost, "rounds_used": r - 1})
            return draft
        with mlflow.start_span(name=f"refine_{r}", span_type=SpanType.LLM) as ref_sp:
            ref_prompt = (
                f"Log:\n{report}\n\nPrevious draft:\n{draft}\n\nCritique:\n{crit}\n\n"
                "Rewrite the 3-sentence RCA incorporating the critique."
            )
            ref_sp.set_inputs({"prompt": ref_prompt, "params": {"model": MODEL, "round": r,
                                                                  "critique": crit, "temperature": TEMPERATURE}})
            draft, c, lat = _chat([
                {"role": "system", "content": sys},
                {"role": "user",   "content": ref_prompt},
            ])
            running_cost = _accumulate_cost(running_cost, c, latency_ms=lat)
            ref_sp.set_attribute("latency_ms", lat)
            ref_sp.set_attribute("cost_usd", c)
            ref_sp.set_outputs({"text": draft, "cost_usd": c, "latency_ms": lat})

    mlflow.update_current_trace(tags={"stop_reason": "max_rounds", "rounds_used": str(max_rounds)})
    if root_span: root_span.set_outputs({"text": draft, "cost_usd": running_cost, "rounds_used": max_rounds})
    return draft

In [ ]:
@mlflow.trace(name="tot_solve", span_type=SpanType.CHAIN,
              attributes={"technique": "tot", "beam": 2, "depth": 2})
def tot_solve(report: str, beam: int = 2, depth: int = 2) -> str:
    """Tree-of-Thoughts: propose `beam` candidates, evaluate each, keep top, expand again.

    Temperature note: propose step uses 0.7 (diversity); evaluate + synthesize use TEMPERATURE (0.0).
    """
    _reset_running_latency()
    root_span = mlflow.get_current_active_span()
    if root_span:
        root_span.set_inputs({"prompt": report,
                              "params": {"model": MODEL, "technique": "tot", "beam": beam, "depth": depth,
                                          "temperature": TEMPERATURE, "propose_temperature": 0.7}})
    if USE_MOCK:
        ans, cost = _mock_answer("tot", report)
        _accumulate_cost(0.0, cost, latency_ms=0.0)
        if root_span: root_span.set_outputs({"text": ans, "cost_usd": cost})
        return ans
    sys = "You are a senior SRE doing RCA from a log snippet."
    running_cost = 0.0
    frontier = [""]  # seed: empty partial answer

    for d in range(depth):
        candidates: list[tuple[float, str]] = []
        for parent in frontier:
            with mlflow.start_span(name=f"propose_d{d}", span_type=SpanType.LLM) as prop_sp:
                prop_prompt = (
                    f"Log:\n{report}\n\nPartial RCA so far:\n{parent or '(none)'}\n\n"
                    f"Propose {beam} distinct next-step continuations toward a 3-sentence final RCA. "
                    "Return them numbered 1) ... 2) ..."
                )
                prop_sp.set_inputs({"prompt": prop_prompt,
                                    "params": {"model": MODEL, "depth": d, "beam": beam, "parent": parent,
                                                "temperature": 0.7}})
                proposal_msg, c, lat = _chat([
                    {"role": "system", "content": sys},
                    {"role": "user",   "content": prop_prompt},
                ], temperature=0.7)
                running_cost = _accumulate_cost(running_cost, c, latency_ms=lat)
                prop_sp.set_attribute("latency_ms", lat)
                prop_sp.set_attribute("cost_usd", c)
                prop_sp.set_outputs({"text": proposal_msg, "cost_usd": c, "latency_ms": lat})
            children = re.findall(r"\d\)\s*(.+?)(?=\n\d\)|$)", proposal_msg or "", re.S)
            children = [ch.strip() for ch in children][:beam] or [proposal_msg or ""]
            for child in children:
                with mlflow.start_span(name=f"evaluate_d{d}", span_type=SpanType.LLM) as eval_sp:
                    eval_prompt = (
                        f"Log:\n{report}\n\nCandidate RCA continuation:\n{child}\n\n"
                        "Score this continuation 0.0–1.0 on: cites a timestamp, identifies a service, proposes mitigation."
                    )
                    eval_sp.set_inputs({"prompt": eval_prompt,
                                        "params": {"model": MODEL, "depth": d, "candidate": child,
                                                    "temperature": TEMPERATURE}})
                    score_msg, c, lat = _chat([
                        {"role": "system", "content": "You are a strict reviewer. Reply with ONLY a float 0.0–1.0."},
                        {"role": "user",   "content": eval_prompt},
                    ])
                    running_cost = _accumulate_cost(running_cost, c, latency_ms=lat)
                    m_ = re.search(r"[01](?:\.\d+)?", score_msg or "0")
                    score = float(m_.group(0)) if m_ else 0.0
                    eval_sp.set_attribute("latency_ms", lat)
                    eval_sp.set_attribute("cost_usd", c)
                    eval_sp.set_outputs({"text": score_msg, "score": score, "cost_usd": c, "latency_ms": lat})
                    eval_sp.set_attribute("score", score)
                candidates.append((score, (parent + "\n" + child).strip()))
        candidates.sort(key=lambda x: -x[0])
        frontier = [c[1] for c in candidates[:beam]]

    # Final synthesis from best partial
    best = frontier[0] if frontier else ""
    with mlflow.start_span(name="synthesize", span_type=SpanType.LLM) as syn_sp:
        syn_prompt = (
            f"Log:\n{report}\n\nBest reasoning trace:\n{best}\n\n"
            "Write the final 3-sentence RCA: service, timestamp, mitigation."
        )
        syn_sp.set_inputs({"prompt": syn_prompt, "params": {"model": MODEL, "best_partial": best,
                                                              "temperature": TEMPERATURE}})
        final, c, lat = _chat([
            {"role": "system", "content": sys},
            {"role": "user",   "content": syn_prompt},
        ])
        running_cost = _accumulate_cost(running_cost, c, latency_ms=lat)
        syn_sp.set_attribute("latency_ms", lat)
        syn_sp.set_attribute("cost_usd", c)
        syn_sp.set_outputs({"text": final, "cost_usd": c, "latency_ms": lat})
    if root_span: root_span.set_outputs({"text": final, "cost_usd": running_cost})
    return final

In [ ]:
TECHNIQUES = {
    "zero_shot_cot":  zero_shot_cot,
    "plan_and_solve": plan_and_solve,
    "self_refine_2r": self_refine,
    "tot_beam2":      tot_solve,
}
print("Registered techniques:", list(TECHNIQUES.keys()))

## Scorers — built-in judges + custom

| Scorer | Type | What it measures |
|---|---|---|
| `Correctness()` | Built-in LLM judge | Output vs `expected_response` |
| `Guidelines("rca_rules")` | Built-in LLM judge | Adheres to our 3-sentence RCA rubric |
| `Safety()` | Built-in LLM judge | No harmful content |
| `cites_evidence` | Custom deterministic | Output contains a timestamp + named service |
| `cost_under_5_cents` | Custom + trace tag | Reads `cost_usd` written by the technique |

In [ ]:
@scorer
def cites_evidence(outputs: str) -> bool:
    """RCA outputs must reference a HH:MM:SS timestamp AND a recognised service name."""
    if not isinstance(outputs, str):
        outputs = str(outputs)
    has_ts      = bool(re.search(r"\d{2}:\d{2}:\d{2}", outputs))
    has_service = bool(re.search(r"\b(api|db|kafka|redis|worker|ingress|deploy|kubelet|payments)\b", outputs, re.I))
    return has_ts and has_service

@scorer
def cost_under_5_cents(outputs: str, trace) -> bool:
    """Trace-derived: fail rows whose accumulated cost_usd tag exceeds $0.05."""
    try:
        tags = (trace.info.tags or {}) if trace is not None else {}
        return float(tags.get("cost_usd", "0")) <= 0.05
    except (ValueError, AttributeError):
        return False

GUIDELINES = (
    "The RCA response must (1) identify the root-cause service, "
    "(2) cite at least one timestamp from the input log, "
    "(3) propose one concrete mitigation."
)
print("Scorers ready.")

## The evaluation loop

One MLflow run per technique. Inside each run, `mlflow.genai.evaluate` does the heavy lifting: it runs `predict_fn` over every eval row, applies every scorer in parallel, logs a trace per row, and aggregates the metrics. No hand-rolled scoring loop.

In [ ]:
all_results = {}
for name, fn in TECHNIQUES.items():
    print(f"\n=== Evaluating: {name} ===")
    with mlflow.start_run(run_name=name):
        mlflow.log_param("technique", name)
        mlflow.log_param("model",     MODEL)
        mlflow.log_param("eval_set",  "rca_golden_v1")
        mlflow.log_param("n_cases",   len(eval_data))

        # `_fn=fn` captures the function in the closure to avoid late-binding bugs.
        predict = (lambda report, _fn=fn: _fn(report))

        results = mlflow.genai.evaluate(
            data=eval_data,
            predict_fn=predict,
            scorers=[
                Correctness(),
                Guidelines(name="rca_rules", guidelines=GUIDELINES),
                Safety(),
                cites_evidence,
                cost_under_5_cents,
            ],
        )
        all_results[name] = results
        for k, v in sorted(results.metrics.items()):
            try:
                print(f"  {k}: {float(v):.3f}")
            except (TypeError, ValueError):
                print(f"  {k}: {v}")

## Open the MLflow Compare view

1. In a terminal: `mlflow ui --port 5000` (already running for this demo).
2. Browser: http://localhost:5000 → experiment **`session6/demo_03_technique_comparison`**.
3. Select all four runs → click **Compare**.
4. In the table view, click the `Correctness/mean` column header to rank techniques.
5. Click any run → **Traces** tab → expand the trace to see:
   - `self_refine` — the generate / critique_1 / refine_1 / critique_2 sub-spans
   - `tot_solve` — the propose_d0 / evaluate_d0 / propose_d1 / evaluate_d1 / synthesize tree
   - `cost_usd` tag on every trace root

In [ ]:
# Find the best run programmatically — same query you would run from CI.
runs_df = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT],
    order_by=["metrics.`correctness/mean` DESC"],
    max_results=20,
)

metric_cols = [c for c in runs_df.columns if c.startswith("metrics.")]
param_cols  = ["params.technique", "params.model"]
show_cols   = [c for c in param_cols + metric_cols if c in runs_df.columns]

print("Top runs ranked by Correctness/mean:\n")
print(runs_df[show_cols].head(4).to_string(index=False))

## The cost / quality frontier

Read the Compare table left-to-right:

- **Plan-and-Solve** is usually the highest accuracy-per-dollar — same call count as Zero-shot CoT, measurable quality lift.
- **Tree-of-Thoughts** can score **+6 pp** on `Correctness/mean` over Plan-and-Solve, but at **40× the cost** and **≈10× the latency**.
- **Self-Refine** sits in between; its value depends entirely on whether the critic has an external signal. Without one, see Misconception 0 in [03-wiki/09-production.md](../../03-wiki/09-production.md).

The right architecture is a business decision, not a technique-preference debate. The MLflow Compare view is the dashboard that surfaces it.

## Takeaways

- **`mlflow.genai.evaluate` replaces hand-rolled eval loops.** One call: run, score, log traces, aggregate metrics.
- **Built-in scorers cover the common cases** — `Correctness`, `Guidelines`, `Safety`, `RelevanceToQuery`, `Completeness`.
- **Custom scorers can read the trace.** `cost_under_5_cents` pulls `cost_usd` from trace tags written during the technique — cost becomes a first-class quality signal.
- **The MLflow Compare view is the technique-selection tool.** Stop arguing about ToT vs CoT. Run the eval. Read the numbers.

Next — demo 4 will use these same MLflow primitives to evaluate a full Plan → Solve → Verify chain with per-step scorers.

## ⚠️ Reasoning model caveat

This notebook assumes a **non-reasoning** base model. On `o1` / `o3` / Claude Extended Thinking / Gemini Thinking:
- Drop the CoT / ToT / Plan-and-Solve scaffolding — the model does it internally
- Use `developer` role instead of `system` on `o1-2024-12-17+`
- Never pass OpenAI `reasoning_effort` and Gemini `thinking_level` through the same OpenAI-compatible adapter
- See Block 5 of the slides for what's redundant vs what survives

The **eval-loop pattern itself survives** — `mlflow.genai.evaluate(...)` with the same scorers will work just fine to compare reasoning models against each other. Only the scaffold-the-prompt techniques on the left collapse to a single "ask the model" call.

## 🎥 Replay every input + output in MLflow

Every prompt and every response from this demo is now in MLflow. Open the UI and click any trace to see the full I/O log.

In [ ]:
import urllib.parse
import IPython.display as D

ui = MLFLOW_TRACKING_URI.rstrip("/")
exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
exp_id = exp.experiment_id if exp else "0"
link = f"{ui}/#/experiments/{exp_id}?searchFilter=&orderByKey=attributes.start_time&orderByAsc=false"
D.display(D.Markdown(f"**Open the experiment in MLflow:** [{link}]({link})"))

# Recent traces (last 5)
recent = mlflow.search_traces(experiment_ids=[exp_id], max_results=5)
recent[["request_id", "timestamp_ms", "execution_time_ms", "tags"]] if not recent.empty else "No traces yet — run the cells above."